In this assignment you will be asked to extend the work by Gatti et al by checking whether form-meaning mappings learned on a different yet related language to that considered in the original study still capture the perceived valence of pseudowords. To do this you will be asked to engage with several different resources and adapt the pipeline following the instructions. Along the way, you will be asked to answer a few questions.

You need to submit the complete notebook in .ipynb format, with intermediate outputs visible. The notebook should be named as follows:

CL2025_groupN_assignment.ipynb

where N is the group number. Submissions in the wrong format or with names not adhering to the guidelines will not be evaluated.

Indicate group members' names, student numbers, and contributions below:
- 1. Aiden Ricksen - 2117510
- 2. Imad-eddine el Bakiouli - 2121984

We worked on both the code and the open questions together, following up on eachothers work throughout the project. We had multiple meetings to work on the project.


In [82]:
# the code has been tested using the psycho-embeddings library to extract representations from LLMs. You can also use other libraries,
# as long as you make sure that you are producing the correct output.
!git clone https://github.com/MilaNLProc/psycho-embeddings.git
%cd psycho-embeddings
!pip install datasets

Cloning into 'psycho-embeddings'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (199/199), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 199 (delta 105), reused 141 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (199/199), 67.91 KiB | 556.00 KiB/s, done.
Resolving deltas: 100% (105/105), done.
/content/psycho-embeddings/psycho-embeddings/psycho-embeddings/psycho-embeddings


In [83]:
# the solution to the assignment has been obtained using these packages.
# you're free to use other packages though: consider this as an indication, not a prescription.
import nltk
import torch
import Levenshtein
import numpy as np
import pandas as pd
import pickle as pkl
import fasttext.util
import fasttext as ft
from tqdm import tqdm
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from collections import defaultdict
from transformers import AutoTokenizer
from sklearn.linear_model import LinearRegression
from psycho_embeddings import ContextualizedEmbedder
from sklearn.feature_extraction.text import CountVectorizer
from transformers import RobertaTokenizer, RobertaForSequenceClassification


**Task 1** (*10 points available, see breakdown per task below*)

You should replicate the main design in the paper *Valence without meaning* by Gatti and colleagues (2024), using estimates collected for Dutch word valence to train linear regression models and apply them to predict the valence of English pseudowords from Gatti and colleagues.

In detail, to train your regression models, you should use the dataset by Speed and Brysbaert (2024) containing crowd-sourced valence ratings (use the metadata to identify the relevant columns) collected for approximately 24,000 Dutch words. See the paper *Ratings of valence, arousal, happiness, anger, fear, sadness, disgust, and surprise for 24,000 Dutch words* by Speed and Brysbaert (2024). Use the dataset available at this link: https://osf.io/h76zj.

You should train a letter unigram model and a bigram model. Each model should be trained on Dutch words only.

Pay attention to one issue though: pseudowords created for English may be valid words in Dutch: therefore, you should first filter the list of pseudowords against a large store of Dutch words. To do so, use the words in the Dutch prevalence lexicon available in this OSF repository: https://osf.io/9zymw/. Essentially, you need to exclude any pseudoword that happens to be a word for which a prevalence estimate is available, whatever the prevalence is.

Each code block indicates how many points are available and how they are attributed.

In [98]:
# read in the pseudowords from Gatti and colleagues, as well as the valence ratings for 24,000 Dutch words from Speed and Brysbaert (2024)
# show the first 5 lines of each dataset.
# 1 point for identifying the correct files and correctly loading their content

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

gattifile = pd.read_csv("/content/drive/MyDrive/Gatti_pseudowords_1500.csv")
print("First 5 lines of the dataset from Gatti:")
print(gattifile.head())
pseudowords = gattifile["pseudoword"]

xlsx_path = '/content/drive/MyDrive/SpeedBrysbaertEmotionNorms.xlsx'
valence = pd.read_excel(xlsx_path)

print("First 5 lines of the data set from Speed and Brysbaert:")
print(valence.head())




Mounted at /content/drive
First 5 lines of the dataset from Gatti:
   X pseudoword     Value  predicted_valence  predictedL_valence  \
0  1     abhert  0.452501           7.414814            5.116167   
1  2     abhict  0.434171           8.233714            5.059183   
2  3     acleat  0.527803           5.552468            5.262971   
3  4     acmure  0.604889           8.714640            5.120029   
4  5      acoed  0.538990           7.340002            5.115652   

   predictedL_Bi_valence  predicted_Dim_valence  predictedL_Dim_valence  \
0               6.444633               6.783771                6.630497   
1               6.509936               7.366068                7.377534   
2               5.245826               5.268643                5.396114   
3               6.562896               7.680827                7.583230   
4               5.309727               7.105662                7.024771   

   predictedBi_Dim_valence  predictedBi_valence  LDist  Ortho_VAL  \
0   

In [99]:
# filter out pseudowords that happen to be valid Dutch words (mind case folding!)
# show the set of pseudowords filtered out.
# 1 point for applying the correct filtering

print("Original pseudoword count:", len(gattifile))
prev = pd.read_csv(
    '/content/drive/MyDrive/prevalence_netherlands.csv',
    sep='\t'
)

valid_words = set(prev['word'].astype(str).str.lower())

pws = gattifile['pseudoword'].astype(str)
pws_low = pws.str.lower()

mask_real = pws_low.isin(valid_words)
removed = set(pws[mask_real])
print("Pseudowords removed:", removed)

filtered_gatti = gattifile.loc[~mask_real].reset_index(drop=True)
print("Remaining pseudoword count:", len(filtered_gatti))

pseudowords_list = filtered_gatti['pseudoword'].str.lower().tolist()



Original pseudoword count: 1500
Pseudowords removed: {'pimpen'}
Remaining pseudoword count: 1499


In [100]:
# encode Dutch words and pseudowords from Gatti et al as uni- and bi-gram vectors
# show the uni-gram and bi-gram encoding of the pseudoword ampgrair
# 2 points for correctly encoding the target strings as uni- and bi-gram vectors

dutch_words = valence["Word"].str.lower().tolist()

uni_vec = CountVectorizer(analyzer="char", ngram_range=(1, 1))
bi_vec  = CountVectorizer(analyzer="char", ngram_range=(2, 2))

X_dutch_uni = uni_vec.fit_transform(dutch_words)
X_dutch_bi  = bi_vec.fit_transform(dutch_words)
X_pseudo_uni = uni_vec.transform(pseudowords_list)
X_pseudo_bi  = bi_vec.transform(pseudowords_list)

ex = ["ampgrair"]
uni_ex = uni_vec.transform(ex).toarray()[0]
bi_ex  = bi_vec.transform(ex).toarray()[0]

print("\nUnigram vector for 'ampgrair':")
print(uni_ex)
print("\nBigram vector for 'ampgrair':")
print(bi_ex)



Unigram vector for 'ampgrair':
[2 0 0 0 0 0 1 0 1 0 0 0 1 0 0 1 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Bigram vector for 'ampgrair':
[0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

In [102]:
# use word valence estimates from Speed and Brysbaert (2024) to train
# - a uni-gram model
# - a bi-gram model
# 2 points for correctly trained models

y = valence["Valence"].values

uni_model = LinearRegression().fit(X_dutch_uni, y)

bi_model = LinearRegression().fit(X_dutch_bi, y)

uni_r2 = uni_model.score(X_dutch_uni, y)
bi_r2 = bi_model.score(X_dutch_bi, y)

print(f"Unigram model R2 score: {uni_r2:.4f}")
print(f"Bigram model R2 score: {bi_r2:.4f}")

Unigram model R2 score: 0.0095
Bigram model R2 score: 0.1160


In [103]:
# apply trained models to predict the valence of pseudowords from Gatti et al (2024).
# Then apply the same models back onto the training set to see how well they predict the valence of words in Speed and Brysbaert (2024).
# 2 points for correctly applied models

y_pseudo_pred_uni = uni_model.predict(X_pseudo_uni)
y_pseudo_pred_bi  = bi_model.predict(X_pseudo_bi)

y_train_pred_uni = uni_model.predict(X_dutch_uni)
y_train_pred_bi  = bi_model.predict(X_dutch_bi)

print("Predicted valence for pseudowords (unigram):", y_pseudo_pred_uni)
print("Predicted valence for pseudowords (bigram) :", y_pseudo_pred_bi)
print("\nPredicted valence for Dutch words (unigram):", y_train_pred_uni)
print("Predicted valence for Dutch words (bigram) :", y_train_pred_bi)

Predicted valence for pseudowords (unigram): [2.91964934 2.93271094 3.00193988 ... 2.83886063 2.84023961 2.79305788]
Predicted valence for pseudowords (bigram) : [3.23914867 3.04290828 3.17845976 ... 2.86050361 3.00660316 3.05633199]

Predicted valence for Dutch words (unigram): [2.97356517 3.04873071 2.99644161 ... 2.96466515 2.95004878 2.90939766]
Predicted valence for Dutch words (bigram) : [3.07687701 2.86048787 3.14367623 ... 2.79148374 2.6557058  2.96608726]


In [121]:
# compute the Spearman correlation coefficients between true valence and predicted valence under both uni- and bi-gram models for
# - words from Speed and Brysbaert (2024)
# - pseudowords from Gatti and colleagues (2024)
# show both correlation coefficients.
# 2 points for the correct Spearman correlation coefficients (rounded to the third decimal place)

r_uni_train, _ = spearmanr(y, y_train_pred_uni)
r_bi_train,  _ = spearmanr(y, y_train_pred_bi)

y_pseudo_true = filtered_gatti["Value"].values

y_pseudo_pred_uni = y_pseudo_pred_uni[:len(y_pseudo_true)]
y_pseudo_pred_bi  = y_pseudo_pred_bi[:len(y_pseudo_true)]

r_uni_pseudo, _ = spearmanr(y_pseudo_true, y_pseudo_pred_uni)
r_bi_pseudo,  _ = spearmanr(y_pseudo_true, y_pseudo_pred_bi)

print(f"Spearman correlation (unigram) of Dutch words:    p = {r_uni_train:.3f}")
print(f"Spearman correlation (bigram)  of Dutch words:    p = {r_bi_train:.3f}")
print(f"Spearman correlation (unigram) of pseudowords:    p = {r_uni_pseudo:.3f}")
print(f"Spearman correlation (bigram)  of pseudowords:    p = {r_bi_pseudo:.3f}")

Spearman correlation (unigram) of Dutch words:    p = 0.090
Spearman correlation (bigram)  of Dutch words:    p = 0.321
Spearman correlation (unigram) of pseudowords:    p = 0.260
Spearman correlation (bigram)  of pseudowords:    p = 0.101


**Task 2** (*8 points available, see breakdown below*)

Again following Gatti and colleagues, you should encode the target strings (pseudowords and Dutch words from Speed and Brysbaert) as fastText embeddings, train a multiple regression model on Dutch words and apply it to the pseudowords in Gatti et al. You should finally report the Spearman correlation coefficient between observed and predicted valence for both words and pseudowords.

You should use the pre-trained fastText model for Dutch, available at this page: https://fasttext.cc/docs/en/crawl-vectors.html

Finally, you should answer two questions about the fastText model (see below).

In [90]:
# load the fastText model
# 1 point for correctly loading the appropriate fastText model
import fasttext.util
ft = fasttext.load_model("/content/drive/MyDrive/cc.nl.300.bin")


What is the dimensionality of the pre-trained Dutch fastText embeddings? The dimensionality of the pre-trained Dutch fasttext embeddings is 300 (*1 point for the correct answer*)

What minimum and maximum n-gram size was specified for training this fastText model? The n-gram that was used for the training of this fasttext model is Lenght 3 to 6 (*1 point for the correct answer*)

In [110]:
# encode Dutch words and pseudowords as fastText embeddings
# show the first 20 values of the embedding of the word 'speelplaats' and of the pseudoword 'danchunk'
# 2 points for correctly encoding words and pseudowords with fastText

X_ft = np.vstack([ft.get_word_vector(w) for w in dutch_words])
X_pw_ft = np.vstack([ft.get_word_vector(w) for w in pseudowords_list])

real_word   = "speelplaats"
pseudo_word = "danchunk"

vec_real   = ft.get_word_vector(real_word)
vec_pseudo = ft.get_word_vector(pseudo_word)

print(f"First 20 values of '{real_word}':\n{vec_real[:20]}\n")
print(f"First 20 values of '{pseudo_word}':\n{vec_pseudo[:20]}")

First 20 values of 'speelplaats':
[ 0.0253247  -0.00634261  0.02746305 -0.04024595  0.04888906  0.00660965
 -0.04152017 -0.01824508 -0.00645641  0.00093806  0.0708492  -0.03291791
  0.00263817 -0.02825846 -0.02188046 -0.03188037 -0.01846142 -0.02203094
 -0.01883078 -0.00259199]

First 20 values of 'danchunk':
[-0.00592199  0.00097547  0.05925412  0.00053251 -0.00386978 -0.02089076
 -0.02829577  0.00972911 -0.02510111 -0.11454885 -0.02695064  0.01551034
  0.02384409  0.01009528  0.04545438  0.00997385 -0.00474529  0.02524533
  0.02430548 -0.02851078]


In [108]:
# train regression model on word valence
# 1 point for correctly training the regression model

ft_reg = LinearRegression().fit(X_ft, y)

print("FastText‐based regression R2 on Dutch valence:", ft_reg.score(X_ft, y))

FastText‐based regression R2 on Dutch valence: 0.5200443073017833


In [118]:
# apply the trained model to predict the valence of pseudowords from Gatti et al (2024).
# Then apply the same model back onto the training set to see how well it predicts the valence of words in Speed and Brysbaert (2024).
# 1 point for correctly applied model

y_pw_pred_ft   = ft_reg.predict(X_pw_ft)
y_train_pred_ft = ft_reg.predict(X_ft)

print("Predicted valence for pseudowords (FastText):", y_pw_pred_ft)
print("Predicted valence for Dutch words (FastText):", y_train_pred_ft)


Predicted valence for pseudowords (FastText): [3.1155238 3.0766866 2.8852634 ... 3.0173347 2.9208107 2.9239726]
Predicted valence for Dutch words (FastText): [4.3099604 2.865531  4.036543  ... 3.1800575 2.8078961 3.1230912]


In [122]:
# compute the Spearman correlation coefficients between true valence and predicted valence for
# - words from Speed and Brysbaert (2024)
# - pseudowords from Gatti and colleagues (2024)
# show the correlation coefficient.
# 1 point for the correct Spearman correlation coefficients (rounded to the third decimal place)

r_dutch, p_dutch = spearmanr(y, y_train_pred_ft)
r_pseudo, p_pseudo = spearmanr(y_pseudo_true, y_pw_pred_ft)

print(f"Spearman correlation of Dutch words:      p = {r_dutch:.3f}")
print(f"Spearman correlation of pseudowords :     p = {r_pseudo:.3f}")


Spearman correlation of Dutch words:      p = 0.724
Spearman correlation of pseudowords :     p = 0.102


**Task 3** (*6 points available, see breakdown below*)

Now you are asked to extend the work by Gatti et al by also considering the representations learned by a transformer-based models, in detail *RobBERT v2* (https://huggingface.co/pdelobelle/robbert-v2-dutch-base). You should follow the same pipeline as for the previous models, encoding both Dutch words from Speed and Brysbaert (2024) and the pseudowords from Gatti et al using the embedding of each string at layer 0, before positional information is factored in. If a string consists of multiple tokens, average the embeddings of all tokens to produce the embedding of the whole string. Then train a multiple regression model on the valence of Dutch words, apply it to the pseudowords, and compute the Spearman correlation between observed and predicted ratings.

Use the HuggingFace model card for RobBERT v2 to check how to access it.

I recommend saving the embeddings to file once you have generated them and you know they are correct: embedding thousands of strings takes some time, and you don't want to have to do it again. For the same reason, develop your code by considering only a small fractions of the words and pseudowords, in order to quickly see if something is wrong. Only when you are positive it works, embed all strings.

In [114]:
# load and instantiate the right model
# 1 point for loading the right model

tokenizer = RobertaTokenizer.from_pretrained("pdelobelle/robbert-v2-dutch-base")
model = RobertaForSequenceClassification.from_pretrained("pdelobelle/robbert-v2-dutch-base")


loading file vocab.json from cache at /root/.cache/huggingface/hub/models--pdelobelle--robbert-v2-dutch-base/snapshots/271b8bf12b7e429434ce953efb432e8373e84453/vocab.json
loading file merges.txt from cache at /root/.cache/huggingface/hub/models--pdelobelle--robbert-v2-dutch-base/snapshots/271b8bf12b7e429434ce953efb432e8373e84453/merges.txt
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at /root/.cache/huggingface/hub/models--pdelobelle--robbert-v2-dutch-base/snapshots/271b8bf12b7e429434ce953efb432e8373e84453/special_tokens_map.json
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--pdelobelle--robbert-v2-dutch-base/snapshots/271b8bf12b7e429434ce953efb432e8373e84453/tokenizer_config.json
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--pdelobelle--robbert-v2-dutch-base/snapshots/271b8bf12b7e429434ce953efb432e8373e84453/tokenizer.json
loading file chat_template.jinja from c

In [115]:
# encode the words and pseudowords using RobBERT v2. I've used the free GPU runtime on COLAB to speed things up,
# but in this case you need to batch the words and pseudowords. You can use the function below to create batches
# but you will have to pay attention at how you store embeddings.
# show the first 20 values of the embedding of the word 'miauwen' and of the pseudoword 'lixthless'
# 2 points for correctly encoding words and pseudowords

def chunks(lst, n):
    """Chunks a list into equal chunks containing n elements. Returns a list of lists."""
    chunked = []
    for i in range(0, len(lst), n):
        chunked.append(lst[i : i + n])
    return chunked

def embed_layer0(texts, batch_size=64):
    embs = []
    for batch in chunks(texts, batch_size):
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            lookup = model.roberta.embeddings.word_embeddings(enc["input_ids"])
        core = lookup[:, 1:-1, :]
        embs.append(core.mean(dim=1))
    return torch.cat(embs, dim=0)

X_rob_dutch = embed_layer0(dutch_words,      batch_size=128)
X_rob_pws   = embed_layer0(pseudowords_list, batch_size=128)

example_vecs = embed_layer0(["miauwen", "lixtheless"], batch_size=2)
print("First 20 values of 'miauwen':  ", example_vecs[0,:20].tolist())
print("First 20 values of 'lixtheless':", example_vecs[1,:20].tolist())


First 20 values of 'miauwen':   [0.01574459858238697, -0.050622452050447464, -0.013598739169538021, 0.008594965562224388, -0.025512464344501495, 0.05335233733057976, 0.07680433243513107, -0.0515216588973999, 0.05702745541930199, 0.015662498772144318, -0.006799475289881229, -0.04346112906932831, 0.006691428832709789, 0.020613234490156174, -0.011536190286278725, 0.06474956125020981, 0.010083496570587158, -0.00346448365598917, 0.024953659623861313, -0.016500618308782578]
First 20 values of 'lixtheless': [-0.018898235633969307, 0.06503724306821823, -0.06651163846254349, 0.04543512314558029, -0.002323267050087452, 0.011083191260695457, 0.00870988517999649, -0.031088024377822876, 0.008913702331483364, -0.02554977312684059, -0.009220123291015625, -0.011074005626142025, -0.07307430356740952, 0.05849800258874893, -0.004880142398178577, -0.03039686381816864, 0.02213442325592041, -0.01936710998415947, 0.07086442410945892, -0.031020576134324074]


In [116]:
# train regression model on word valence estimates from Speed and Brysbaert (2024)
# 1 point for correctly training the regression model

rob_reg = LinearRegression().fit(X_rob_dutch, y)

print("RobBERT regression R² on Dutch valence:", rob_reg.score(X_rob_dutch, y))

RobBERT regression R² on Dutch valence: 0.29015233642237737


In [119]:
# apply the trained model to predict the valence of pseudowords from Gatti et al (2024).
# Then apply the same model back onto the training set to see how well it predicts the valence of words in Speed and Brysbaert (2024).
# 1 point for correctly applied model

y_pw_pred_rob    = rob_reg.predict(X_rob_pws.numpy())
y_train_pred_rob = rob_reg.predict(X_rob_dutch.numpy())

print("Predicted valence for pseudowords:", y_pw_pred_rob)
print("Predicted valence for Dutch words:", y_train_pred_rob)



Predicted valence for pseudowords: [2.9676025 2.8038704 2.6597714 ... 3.0554996 2.9347763 2.6780963]
Predicted valence for Dutch words: [3.2607572 3.0649645 3.0775404 ... 2.7749772 2.9288225 3.0169504]


In [123]:
# compute the Spearman correlation coefficients between true valence and predicted valence for
# - words from Speed and Brysbaert (2024)
# - pseudowords from Gatti and colleagues (2024)
# show the correlation coefficient
# 1 point for the correct Spearman correlation coefficients (rounded to the third decimal place)

r_rob_train, _ = spearmanr(y, y_train_pred_rob)
r_rob_pws, _  = spearmanr(y_pseudo_true, y_pw_pred_rob)

print(f"Spearman correlation of Dutch words:    ρ = {r_rob_train:.3f}")
print(f"Spearman correlation of pseudowords:    ρ = {r_rob_pws:.3f}")


Spearman correlation of Dutch words:    ρ = 0.515
Spearman correlation of pseudowords:    ρ = 0.169


**Task 4** (*16 points available, 4 for each question*)

Answer the following questions.

**4a.** Describe the performance of each featurization, comparing
- the performance of a same model between the training and test set
- the performance of different models on the training set
- the performance of different models on the test set

(*4 points available, max 150 words*)

In our training set, the unigram model explains almost no variance (R² = 0.0095, ρ = 0.090) yet generalizes best to pseudowords (ρ = 0.260). Bigrams capture more orthographic structure (R2 = 0.1160, ρ = 0.321) but drop to ρ ≈ 0.101 on novel strings. FastText subword embeddings dominate in-sample (R2 = 0.5200, ρ = 0.724) but collapse to ρ ≈ 0.102 out-of-sample. RobERT layer-0 embeddings sit in between (R2 = 0.2982, ρ = 0.515), with moderate generalization (ρ ≈ 0.169).

Across models on training words, performance ranks 1. fastText, 2. RobERT, 3. bigrams, 4. unigrams. On pseudowords, the order flips: 1. unigrams, 2. RobERT, 3. fastText & bigrams. Thus, while complex subword representations fit real-word valence extremely well, they overfit and struggle with novel letter strings, whereas simple letter-frequency cues transfer most robustly.  


**4b.** Compare the correlations you found when training uni-gram, bi-gram, and fastText models on Dutch words and the correlations of similar models trained on English data as reported by Gatti and colleagues; summarize the most important similarities and differences.
(*4 points available, max 150 words*)

The broad pattern you see in Dutch closely mirrors the English results from Gatti et al. (2023). In English, a linear model on single letters achieved only r = .11, adding bigrams raised this to r = .33, and a full fastText‐enriched model reached r = .80. In Dutch, we likewise find very low correlation for unigrams (ρ = .09), a strong jump for bigrams (ρ = .32), and the highest performance with fastText (ρ = .72).

Thus, in both languages: (1) letter‐only features capture almost no valence, (2) orthographic context (bigrams) yields a medium effect, and (3) subword‐informed embeddings deliver the largest boost. The main quantitative difference is that all three Dutch correlations are slightly lower than their English counterparts—especially fastText (ρ = .72 vs. r = .80)—suggesting somewhat less predictive regularity in our Dutch valence norms.

**4c.** Do you think the performance of the fastText featurization would change if you were to use different n-grams? Would you make them smaller or larger? Justify your answer.

(*4 points available, max 150 words*)

FastText uses small parts of words called subword n-grams, usually 3 to 6 letters long. These help the model fit known words well, but they don’t work as well on made-up words (pseudowords).

If we use shorter n-grams (like 3 to 4), the model might handle new, made-up words better. But this could slightly reduce how well it fits real words.

If we use longer n-grams (like 3 to 8), the model might understand real words even better—especially in a language like Dutch—but it may become worse at handling new words because of overfitting.

Since FastText struggles with made-up words in our case, using slightly shorter n-grams could give a better balance between accuracy on known words and generalizing to new ones.


**4d.** Do you think that training the same models on uni-grams, bi-grams, fastText and transformer-based embeddings but using valence ratings for Finnish (a language which uses the same alphabet as English but is not a IndoEuropean language) words would yield a similar pattern of results? Justify your answer.

(*4 points available, max 150 words*)

I’d expect the model ranking to stay the same in Finnish: fastText would still perform best, followed by transformers, then bigrams, and finally unigrams. This is because subword and context-based models capture much more useful information than just counting letters.

However, the differences between models would likely be bigger in Finnish. Finnish words are long and built from many parts (due to its complex grammar), so fastText might do even better at predicting real words—but it could also struggle more with made-up words.

Transformer models, like RobERTa, can understand full word context and would likely handle these complex forms better than fastText.

Meanwhile, bigrams would have a hard time with the complicated spelling patterns in Finnish, and unigrams would still perform poorly. So, we’d probably see the same ranking, but the better models would pull further ahead, and the weaker models would do even worse.

**Task 5** (*3 points available*)

Compute the average Levenshtein Distance (aLD) between each pseudoword and the 20 words at the smallest edit distance from it. Consider the set of words you used to filter out pseudowords that happen to be valid Dutch words (the file is available in this OSF repository: https://osf.io/9zymw/) to retrieve the 20 words at the smallest edit distance.

In [124]:
# compute the average Levenshtein distance from each pseudoword to the words used to filter out pseudowords.
# Show the aLD estimate for the pseudowords 'nedukes', 'pewbin', and 'vibcines'
# 3 points for correctly computing aLD for pseudowords

targets = ['nedukes', 'pewbin', 'vibcines']

for pw in targets:
    dists = [Levenshtein.distance(pw, w) for w in valid_words]
    nearest20 = sorted(dists)[:20]
    aLD = sum(nearest20) / 20
    print(f"{pw}: aLD = {aLD:.3f}")


nedukes: aLD = 2.900
pewbin: aLD = 2.950
vibcines: aLD = 3.550


**Task 6** (*3 points available*)

For each pseudoword, record the number of tokens in which RobBERT v2 encodes it.

In [125]:
# record the number of tokens in which RobBERT divides each pseudoword
# show the number of tokens for the pseudowords 'yuxwas', 'skibfy', and 'errords'
# 3 points for correctly mapping pseudowords to number of tokens

pseudowords = ["yuxwas", "skibfy", "errords"]

for pw in pseudowords:
    token_ids = tokenizer.encode(pw, add_special_tokens=False)
    tokens    = tokenizer.convert_ids_to_tokens(token_ids)
    print(f"{pw:8s} → {len(tokens)} tokens: {tokens}")

yuxwas   → 3 tokens: ['y', 'ux', 'was']
skibfy   → 4 tokens: ['sk', 'ib', 'f', 'y']
errords  → 3 tokens: ['er', 'ror', 'ds']


**Task 7** (*5 points available, see breakdown below*)

Compute the residuals of the predicted valence under the four regressors trained and applied in tasks 2 to 4. Then, correlate the residuals from all four models with aLD. Finally, correlate the residuals from the RobBERT v2 model with the number of tokens in which each pseudoword is split. Use the Pearson's correlation coefficient.

In [126]:
# compute the residuals from all four regression models fitted before
# 1 point available for correctly computing residuals
y_true = filtered_gatti["Value"].values

y_uni = y_pseudo_pred_uni[: len(y_true)]
y_bi  = y_pseudo_pred_bi[: len(y_true)]
y_ft  = y_pw_pred_ft[:   len(y_true)]
y_rob = y_pw_pred_rob[:  len(y_true)]

res_uni = y_true - y_uni
res_bi  = y_true - y_bi
res_ft  = y_true - y_ft
res_rob = y_true - y_rob

In [127]:
# compute the Pearson's correlation between residuals and average LD for all models,
# as well as the correlation between RobBERT v2 residuals and the number of tokens in which each pseudoword
#    is encoded by the RobBERT v2 model.
# show all correlation coefficients
# 4 points for the correct correlation coefficients

aLD_list = []
for pw in pseudowords_list:
    dists = [Levenshtein.distance(pw, w) for w in valid_words]
    nearest20 = sorted(dists)[:20]
    aLD_list.append(sum(nearest20) / 20)
aLD_array = np.array(aLD_list)

token_counts_list = [
    len(tokenizer.encode(pw, add_special_tokens=False))
    for pw in pseudowords_list
]
token_counts = np.array(token_counts_list)


for name, res in zip(
    ["Unigram", "Bigram", "fastText", "RobERT"],
    [res_uni, res_bi, res_ft, res_rob]
):
    r, p = pearsonr(res, aLD_array)
    print(f"{name:8s} residual vs aLD → r = {r:.3f}, p = {p:.3f}")
r_tok, p_tok = pearsonr(res_rob, token_counts)
print(f"RobERT residual vs tokens → r = {r_tok:.3f}, p = {p_tok:.3f}")

Unigram  residual vs aLD → r = -0.117, p = 0.000
Bigram   residual vs aLD → r = -0.109, p = 0.000
fastText residual vs aLD → r = -0.094, p = 0.000
RobERT   residual vs aLD → r = -0.017, p = 0.509
RobERT residual vs tokens → r = 0.120, p = 0.000


**Task 8** What is the relation between the errors each model made and aLD? what about the number of tokens (limited to the RobBERT v2 model)?

(*4 points available, max 150 words*)

The unigram, bigram, and fastText models each show a significant negative correlation between their residuals and aLD (r ≈ –.117, –.109, –.049; p < .001), meaning that the farther a pseudoword is from any real Dutch word, the more those models tend to overestimate its valence. RobERT v2, however, has no meaningful relationship with aLD (r = –.017; p = .51), suggesting its errors aren’t driven by simple orthographic unfamiliarity. Finally, within RobERT, residuals correlate positively with the number of subword tokens (r = .120; p < .001): pseudowords split into more tokens tend to have their valence underestimated.